1. Load the Dataset and Inspect Samples & Classes

In [5]:
import os
# Install via pip first if not already installed: !pip install opendatasets
import pandas as pd

# This will prompt you for your Kaggle Username and Key on the first run
import kagglehub
path = kagglehub.dataset_download("msambare/fer2013")

print(os.listdir(path))
train_dir = os.path.join(path, 'train')
test_dir = os.path.join(path, 'test')

print('Train directory:', train_dir)
print('Test directory:', test_dir)

100%|██████████| 60.3M/60.3M [00:04<00:00, 15.1MB/s]

Extracting files...


['train', 'test']
Train directory: /root/.cache/kagglehub/datasets/msambare/fer2013/versions/1/train
Test directory: /root/.cache/kagglehub/datasets/msambare/fer2013/versions/1/test


2. Identify Data Imbalance

In [7]:
import os

# Count samples for each emotion class in the train directory
class_counts = {}
for class_name in sorted(os.listdir(train_dir)):
  class_path = os.path.join(train_dir, class_name)
  if os.path.isdir(class_path):
    class_counts[class_name] = len(os.listdir(class_path))

print('Train Class Distribution:')
for emotion, count in class_counts.items():
  print(f'- {emotion}: {count} samples')

Train Class Distribution:
- angry: 3995 samples
- disgust: 436 samples
- fear: 4097 samples
- happy: 7215 samples
- neutral: 4965 samples
- sad: 4830 samples
- surprise: 3171 samples


3. Convert Pixel String to Image Array and Reshape

In [8]:
from tensorflow.keras.utils import img_to_array, load_img

# Pick the first class folder and the first image inside it
sample_class = os.listdir(train_dir)[0]
sample_class_dir = os.path.join(train_dir, sample_class)
sample_img_name = os.listdir(sample_class_dir)[0]
sample_img_path = os.path.join(sample_class_dir, sample_img_name)

# Load the image in grayscale with the expected FER2013 dimensions (48x48)
img = load_img(sample_img_path, color_mode='grayscale', target_size=(48, 48))
img_array = img_to_array(img)

print(f'Confirmed image shape: {img_array.shape}')

Confirmed image shape: (48, 48, 1)


4. Normalization and 80-10-10 Train/Validation/Test Split

In [9]:
import tensorflow as tf

batch_size = 64

# 1. Load 80% of train folder for training
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(48, 48),
    color_mode='grayscale',
    batch_size=batch_size,
)

# 2. Load 20% of train folder for validation
val_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(48, 48),
    color_mode='grayscale',
    batch_size=batch_size,
)

# 3. Load the test dataset from the test folder
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=(48, 48), color_mode='grayscale', batch_size=batch_size
)

# 4. Normalize pixel values to [0, 1] using a Rescaling layer
normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))

print('Datasets successfully loaded and normalized!')

Found 28709 files belonging to 7 classes.
Using 22968 files for training.
Found 28709 files belonging to 7 classes.
Using 5741 files for validation.
Found 7178 files belonging to 7 classes.
Datasets successfully loaded and normalized!


5. Addressing the Class Imbalance for the 'Disgust' Class

In [10]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Reconstruct labels list based on directory counts to compute weights
y_train_labels = []
class_names = sorted(
    [c for c in os.listdir(train_dir) if not c.startswith('.')]
)

for idx, class_name in enumerate(class_names):
  class_path = os.path.join(train_dir, class_name)
  if os.path.isdir(class_path):
    num_files = len(os.listdir(class_path))
    y_train_labels.extend([idx] * num_files)

# Compute balanced class weights
class_weights = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels
)
class_weight_dict = dict(enumerate(class_weights))

print('Calculated Class Weights:', class_weight_dict)

Calculated Class Weights: {0: np.float64(1.0266046844269623), 1: np.float64(9.406618610747051), 2: np.float64(1.0010460615781582), 3: np.float64(0.5684387684387684), 4: np.float64(0.8260394187886635), 5: np.float64(0.8491274770777877), 6: np.float64(1.293372978330405)}
